# M4 — TDD: Fix the Bug, Then Build the Validator
**Duration:** 65 min | **Participant notebook:** `M4_TDD_VALIDATOR.ipynb`

---

## At a glance

| | |
|---|---|
| **Copilot skills taught** | Agent mode, TDD, Plan mode, `/review-cet-function` |
| **Key files** | `curve_pipeline/cet_converter.py` (fixed in Part A), `curve_pipeline/validator.py` (built in Part B/C), `tests/test_cet_converter.py`, `tests/test_validator.py` |
| **What participants produce** | `tests/test_cet_converter.py` — 3 tests for the converter. `tests/test_validator.py` — 7+ tests for the validator. Both files green with 0 regressions. |

**Reference solutions (do not share):**
- `facilitator/validator_solution.py` — full implementation
- `facilitator/test_validator_solution.py` — 17 tests

**Dependency:** Both `cet_converter.py` (fixed) and `validator.py` (implemented) must be green before M5.

**Structure:** Part 0 (agent safety, 5 min) · Part A (TDD fix cet_converter, 15 min) · Part B (Plan + TDD validator, 30 min) · Part C (Implement + review, 15 min)

## Key teaching moment

**Fix the converter first — with tests. Then build the validator on top of something that works.**

Deliver at the opening of Part A:

> "In M2 you fixed the bug on a throwaway branch. The fix disappeared. Today you fix it properly — test first. Before writing a single line of fix, you will have a test that fails because of the bug. That test is what makes the fix permanent."

At Part B opening:

> "The converter is now correct. `CurveValidator` is different — it doesn't fix or compute anything. It checks that the rules were followed. The test you wrote for the winter converter case — `cetStarttime == startTime` — that same assertion is now a validator test too. But now it's a check, not a fix."

At the closing question:

> "What is the difference between `test_cet_converter.py` and `test_validator.py`? One tests that the computation is correct. The other tests that the rules are enforced. You need both."

## Questions and answers

**Part A — after writing failing tests:**

Q: "Why does the winter test pass with the copy-through bug still in place?"

A: Because in winter, CET equals local time — copying `startTime` to `cetStarttime` produces the correct value. The bug is invisible in winter. That is why the February checkpoint gave false confidence.

---

**Part A — after fixing the converter:**

Q: "All three regressions show 0. What does that mean?"

A: The converter is now correct for all three regimes. The tests in `test_cet_converter.py` will catch any future regression — if someone reintroduces the copy-through bug, the summer test fails immediately.

---

**Part B — the key question:**

Q: "Which of your validator tests would pass even if `cet_converter.py` still had the copy-through bug?"

A: The winter test — `cetStarttime == startTime` is what the buggy code produces in winter. Valid test, wrong for the wrong reason. This is why Part A (fixing the converter) must come before Part B (building the validator).

---

**Closing:**

Q: "What is the difference between `test_cet_converter.py` and `test_validator.py`?"

A: `test_cet_converter.py` tests that the computation is correct — given input, does the converter produce the right output? `test_validator.py` tests that the rules are enforced — given output, does the validator correctly identify what is valid and what is not? One tests the producer, the other tests the gatekeeper.

## Watch for

**Participants who skip Part A and go straight to the validator.**

What it looks like: They start on the validator plan without fixing the converter first.

Response: Stop them. "Run the summer regression first. What does it show?" (75 mismatches.) "You are building a validator on top of a broken converter. Fix the converter first."

---

**Copilot generates the converter fix without being asked (in Part A Step 1).**

What it looks like: The test prompt returns both tests and a fix.

Response: Accept the tests, reject the fix. "Tests only. Close the diff on `cet_converter.py`. Run the tests — they should fail."

---

**Tests that pass with the buggy `cet_converter.py`.**

What it looks like: All converter tests pass immediately.

Response: Ask: "Does your summer test verify that `cetStarttime` is one hour behind `startTime`? Or that they are equal?" If equal, the test tests the bug.

---

**Part B validator tests don't distinguish from converter tests.**

What it looks like: `test_validator.py` is essentially a copy of `test_cet_converter.py`.

Response: Ask: "What does the validator add that the converter tests don't cover?" Answer: error/warning detection for invalid types, bad quotes, order B on wrong day. The CET regime tests overlap intentionally — but the validator tests cover more.

## Timing notes

| Segment | Planned | Notes |
|---|---|---|
| Part 0 — agent safety | 5 min | Keep short |
| Part A Step 1 — write failing tests | 5 min | Fast; the key is they fail on summer |
| Part A Step 2 — fix converter | 5 min | Agent prompt is tight; fix is small |
| Part A Step 3 — regression arc | 5 min | Let all three run; 0/0/0 is the payoff |
| Part B Step 1 — plan | 10 min | Review plan for three regimes |
| Part B Step 2 — write validator tests | 20 min | "Which test passes with the bug?" takes 5–8 min |
| Part C — implement + review | 15 min | Fast with test file as spec |

**The non-negotiable:** Every participant must leave M4 with:
1. `pytest tests/test_cet_converter.py` — green
2. `pytest tests/test_validator.py` — green
3. All three regressions — 0 mismatches

If anyone is behind at 12:00, pair them up or give them the solution.

**Lunch gate:** Run `pytest tests/` and all three regressions as a group before breaking.